## Collect & Prepare Your Own Documents

Task:

1. Create a folder named my_docs/.
2. Place at least 3 text-based files in it — .txt, .md, or extracted .pdf text.
  
Examples:
  - A research paper summary you wrote
  - A blog article about AI or sustainability
  - Course notes or company reports

3. Load them into Python.

In [8]:
import os

folder = "my_docs"
os.makedirs(folder, exist_ok=True)

# Create sample corpus if folder is empty
if len(os.listdir(folder)) == 0:

    with open(os.path.join(folder, "data_modelling.txt"), "w", encoding="utf-8") as f:
        f.write("""
Data modelling is the process of designing data structures and defining relationships between data entities.
It helps organizations understand, organize, and manage information efficiently.
The three major types of data models are conceptual, logical, and physical models.
A well-designed data model improves data quality, consistency, and system performance.
Data modelling is an essential step in database development and analytics projects.
""")

    with open(os.path.join(folder, "database_design.txt"), "w", encoding="utf-8") as f:
        f.write("""
Database design focuses on organizing data into tables and defining relationships between them.
Normalization reduces redundancy and improves data integrity.
Primary keys uniquely identify records, while foreign keys establish relationships between tables.
Poor database design can lead to data inconsistency and performance issues.
Relational database systems commonly use structured schemas to manage information.
""")

    with open(os.path.join(folder, "big_data_analytics.txt"), "w", encoding="utf-8") as f:
        f.write("""
Big data refers to datasets that are too large or complex for traditional processing systems.
Organizations use big data analytics to discover patterns, trends, and insights.
Challenges include data storage, processing speed, security, and data integration.
Data warehouses and data lakes are common solutions for managing large-scale data.
Analytics techniques help organizations make data-driven decisions.
""")

documents = []

for file in os.listdir(folder):
    if file.endswith((".txt", ".md")):
        with open(os.path.join(folder, file), "r", encoding="utf-8") as f:
            documents.append(f.read())

print(f"Loaded {len(documents)} documents.\n")

for i, doc in enumerate(documents, 1):
    print(f"Document {i}:")
    print(doc[:200])
    print("-" * 80)

Loaded 3 documents.

Document 1:

RAG stands for Retrieval-Augmented Generation. 
It combines information retrieval with text generation.
RAG helps language models answer using external documents instead of only using stored model kn
--------------------------------------------------------------------------------
Document 2:

AI ethics focuses on fairness, privacy, transparency, accountability, and bias.
Ethical AI systems should avoid discrimination and explain their decisions clearly.
AI models can create risks when the
--------------------------------------------------------------------------------
Document 3:

Transformers are deep learning models used in modern NLP.
They use self-attention to understand relationships between words in a sequence.
BERT, GPT, and many large language models are based on trans
--------------------------------------------------------------------------------


## Chunk Your Texts

**Goal**: Break long text into manageable pieces for retrieval.

In [9]:
def chunk_text(text, chunk_size=200):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)

    return chunks

# TODO: Apply chunking to all your documents
# Try different chunk sizes (100, 200, 400).
# Which size produced more relevant retrievals later? Why?

chunks = []

for doc in documents:
    chunks.extend(chunk_text(doc, chunk_size=200))

print("Total chunks created:", len(chunks))

for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}:")
    print(chunk[:300])

Total chunks created: 3

Chunk 1:
RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation. RAG helps language models answer using external documents instead of only using stored model knowledge. This reduces hallucination and makes answers more grounded.

Chunk 2:
AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discrimination and explain their decisions clearly. AI models can create risks when they are trained on biased or poor-quality data.

Chunk 3:
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.


## Build Your Own Retriever (Semantic)

**Goal**: Retrieve the most relevant chunks using embeddings.

In [10]:
from sentence_transformers import SentenceTransformer, util
import torch

embedder = SentenceTransformer("all-MiniLM-L6-v2")
# TODO: Embed your chunks
chunk_embeddings = embedder.encode(chunks, convert_to_tensor=True)

def retrieve_chunks(query, k=2):
    query_embed = embedder.encode(query, convert_to_tensor=True)

    scores = util.cos_sim(query_embed, chunk_embeddings)[0]

    top_k = torch.topk(scores, k)

    retrieved_chunks = []

    for index in top_k.indices:
        retrieved_chunks.append(chunks[index])

    return retrieved_chunks


print(retrieve_chunks("What is discussed about AI ethics?", k=2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

['AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discrimination and explain their decisions clearly. AI models can create risks when they are trained on biased or poor-quality data.', 'Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.']


## Connect Retrieval with Generation

**Goal**: Use retrieved chunks as context for text generation.

In [11]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2",
    max_new_tokens=120,
    temperature=0.3,
    pad_token_id=50256
)

def mini_rag(query, k=2):
    retrieved = retrieve_chunks(query, k)
    context = "\n".join(retrieved)

    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question:
{query}

Answer:
"""

    result = generator(prompt)[0]["generated_text"]
    return result

# TODO: Try 2–3 questions based on your own documents
#Observe how the generator uses their context.
# Run the same question with and without retrieval.
# Compare and describe differences.
print(mini_rag("Summarize the main idea from my document."))

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation. RAG helps language models answer using external documents instead of only using stored model knowledge. This reduces hallucination and makes answers more grounded.
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.

Question:
Summarize the main idea from my document.

Answer:
The main idea is to use the following context to answer the question.
Question:
The main idea is to use the following context to answer the question.
Answer:
The main idea is to use the following context to answer the question.
Question:
The main idea is to use the following context to answer the question.
Question:
The main idea is to use the following context to answer the question.
Qu

In [12]:
questions = [
    "What is RAG?",
    "What are the main points in AI ethics?",
    "How do transformers work?"
]

for question in questions:
    print("\nQuestion:", question)
    print("-" * 50)
    print(mini_rag(question, k=2))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: What is RAG?
--------------------------------------------------


[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation. RAG helps language models answer using external documents instead of only using stored model knowledge. This reduces hallucination and makes answers more grounded.
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.

Question:
What is RAG?

Answer:
RAG is a combination of the three main components of RAG:
RAG is a combination of the three main components of RAG:
RAG is a combination of the three main components of RAG:
RAG is a combination of the three main components of RAG:
RAG is a combination of the three main components of RAG:
RAG is a combination of the three main components of RAG:
RAG is a combination of the three main components of RAG:
RAG is a comb

[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Use the following context to answer the question.

Context:
AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discrimination and explain their decisions clearly. AI models can create risks when they are trained on biased or poor-quality data.
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.

Question:
What are the main points in AI ethics?

Answer:
AI ethics is a complex and complex problem. It is not a simple problem, but it is a complex problem.
Question:
What is the main point in AI ethics?
Answer:
AI ethics is a complex problem. It is not a simple problem, but it is a complex problem.
Question:
What is the main point in AI ethics?
Answer:
AI ethics is a complex problem. It is not a simple problem, but it is a complex problem.
Question:
What is the main

In [13]:
query = "What are the main points in AI ethics?"

# With retrieval
with_retrieval = mini_rag(query, k=2)

# Without retrieval
no_context_prompt = f"""
Answer the question.

Question:
{query}

Answer:
"""

without_retrieval = generator(no_context_prompt)[0]["generated_text"]

print("With Retrieval:")
print(with_retrieval)

print("\nWithout Retrieval:")
print(without_retrieval)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


With Retrieval:

Use the following context to answer the question.

Context:
AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discrimination and explain their decisions clearly. AI models can create risks when they are trained on biased or poor-quality data.
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.

Question:
What are the main points in AI ethics?

Answer:
AI ethics is a complex and complex system. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and complex. It is complex, complex and compl

## Add a LangChain RAG Chain

**Goal**: Chain the retrieval and generation steps modularly.

In [15]:
import sys
!{sys.executable} -m pip install langchain-huggingface
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

prompt = ChatPromptTemplate.from_template("""
Context:
{context}

Question:
{question}

Answer clearly and concisely:
""")

rag_chain = (
    {
        "context": lambda q: "\n".join(retrieve_chunks(q, 2)),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

# TODO: Run your chain
# Modify the retriever to return top 3 instead of 2.
# Print retrieved chunks before answering.
# Discuss if adding more chunks improved or worsened quality.
print(rag_chain.invoke("What are the key challenges mentioned in AI ethics?"))

[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Human: 
Context:
AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discrimination and explain their decisions clearly. AI models can create risks when they are trained on biased or poor-quality data.
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.

Question:
What are the key challenges mentioned in AI ethics?

Answer clearly and concisely:
AI ethics is a complex, complex and complex problem. It is not a simple problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem. It is a complex problem.
Question:
What is the key challenge to AI ethics?
Answer clearly and concisely:
AI ethics is 

In [16]:
query = "What are the key challenges mentioned in AI ethics?"

retrieved_top3 = retrieve_chunks(query, k=3)

print("Retrieved Top 3 Chunks:")
print("-" * 50)

for i, chunk in enumerate(retrieved_top3, start=1):
    print(f"\nChunk {i}:")
    print(chunk)

context_top3 = "\n".join(retrieved_top3)

prompt_top3 = f"""
Context:
{context_top3}

Question:
{query}

Answer clearly:
"""

answer_top3 = generator(prompt_top3)[0]["generated_text"]

print("\nAnswer using top 3 chunks:")
print(answer_top3)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Retrieved Top 3 Chunks:
--------------------------------------------------

Chunk 1:
AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discrimination and explain their decisions clearly. AI models can create risks when they are trained on biased or poor-quality data.

Chunk 2:
Transformers are deep learning models used in modern NLP. They use self-attention to understand relationships between words in a sequence. BERT, GPT, and many large language models are based on transformer architecture.

Chunk 3:
RAG stands for Retrieval-Augmented Generation. It combines information retrieval with text generation. RAG helps language models answer using external documents instead of only using stored model knowledge. This reduces hallucination and makes answers more grounded.

Answer using top 3 chunks:

Context:
AI ethics focuses on fairness, privacy, transparency, accountability, and bias. Ethical AI systems should avoid discriminatio

## Evaluation & Reflection

### Goal
Critically assess and document the performance of **your own RAG system**.  
Reflect on how your retrieval and generation pipeline behaved with your chosen documents.

---

### Evaluation Questions

| **Question** | **Students Write** |
|---------------|--------------------|
| **How many documents did you use?** | I used 3 documents. |
| **What type of content (topic/domain)?** | I used NLP-related content: RAG, AI ethics, and transformers. |
| **Which retrieval size (chunk length, `top_k`) worked best?** | Top_k = 2 worked better for me because it gave enough useful context without adding too much unrelated text. |
| **Did the model produce hallucinations? When?** |  Yes, sometimes the model produced repeated or unclear answers. This happened mainly during text generation because Distil GPT2 is a small model.|
| **What improvement would you try next?** |  I would try a better model like FLAN-T5 and compare the output with DistilGPT2.|

---

### Home Assignment

- **Try another model**, e.g. `google/flan-t5-base` or `facebook/bart-large`, and compare outputs.  
- **Visualize cosine similarity scores** between your query and retrieved chunks as a bar chart to better understand retrieval ranking.

---




1. Model Comparison

        I compared DistilGPT2 with FLAN-T5.

        DistilGPT2 did not give a clear answer and mostly repeated the prompt.
        FLAN-T5 gave a better and more direct answer.
        For the AI ethics question, FLAN-T5 answered: fairness, privacy, transparency, accountability, and bias.

        So, FLAN-T5 worked better for this RAG task.

2. Cosine Similarity Visualization

        I also visualized cosine similarity scores between the query and retrieved chunks using a bar chart.

        For the query about AI ethics, the scores were:

        Chunk 1: 0.787
        Chunk 2: 0.042
        Chunk 3: 0.190

        Chunk 1 had the highest score, which is correct because it contains the AI ethics content. This shows that semantic retrieval worked properly.

In [ ]:
from transformers import pipeline

flan_generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small",
    max_new_tokens=80
)

def mini_rag_flan(query, k=2):
    retrieved = retrieve_chunks(query, k)
    context = "\n".join(retrieved)

    prompt = f"""
Answer the question using only the context.

Context:
{context}

Question:
{query}

Answer:
"""

    result = flan_generator(prompt)[0]["generated_text"]
    return result


query = "What are the main points in AI ethics?"

print("DistilGPT2 Output:")
print(mini_rag(query, k=2))

print("\nFLAN-T5 Output:")
print(mini_rag_flan(query, k=2))

**Visualize Cosine Similarity Scores**

In [ ]:
import matplotlib.pyplot as plt
from sentence_transformers import util

query = "What are the main points in AI ethics?"

query_embedding = embedder.encode(query, convert_to_tensor=True)

scores = util.cos_sim(query_embedding, chunk_embeddings)[0]

scores_list = scores.cpu().tolist()

chunk_labels = []

for i in range(len(chunks)):
    chunk_labels.append(f"Chunk {i+1}")

plt.figure(figsize=(8, 4))
plt.bar(chunk_labels, scores_list)
plt.title("Cosine Similarity Scores for Retrieved Chunks")
plt.xlabel("Chunks")
plt.ylabel("Cosine Similarity Score")
plt.show()

for i, score in enumerate(scores_list, start=1):
    print(f"Chunk {i} score: {round(score, 3)}")